# OTTO — Training, Temporal Validation & Baselines

This notebook prepares a local OTTO development subset, creates a temporal validation split, defines ranking metrics, and benchmarks:

- Global popularity
- Event-weighted popularity
- Recency-weighted popularity
- Co-visitation

Expected raw file: `data/raw/train.jsonl` or `data/raw/otto-recsys-train.jsonl`.


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import json
import math
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
pd.set_option("display.max_columns", None)


## 1. Locate training data

In [ ]:
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

candidates = [
    RAW_DIR / "train.jsonl",
    RAW_DIR / "otto-recsys-train.jsonl",
]

TRAIN_PATH = next((p for p in candidates if p.exists()), None)

if TRAIN_PATH is None:
    raise FileNotFoundError("Expected train.jsonl or otto-recsys-train.jsonl under data/raw")

print(TRAIN_PATH)
print(f"{TRAIN_PATH.stat().st_size / 1024**3:.2f} GB")


## 2. Load a development subset

In [ ]:
N_SESSIONS = 300_000

sessions = []
with TRAIN_PATH.open("r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= N_SESSIONS:
            break
        sessions.append(json.loads(line))

print(f"Loaded {len(sessions):,} sessions")


## 3. Flatten events

In [ ]:
rows = []

for session_record in sessions:
    sid = session_record["session"]
    for position, event in enumerate(session_record["events"]):
        rows.append({
            "session": sid,
            "aid": event["aid"],
            "ts": event["ts"],
            "type": event["type"],
            "position": position,
        })

events = pd.DataFrame(rows)
events["datetime"] = pd.to_datetime(events["ts"], unit="ms", utc=True)

print(f"Events: {len(events):,}")
print(f"Sessions: {events['session'].nunique():,}")
print(f"Items: {events['aid'].nunique():,}")
events.head()


## 4. Temporal session split

In [ ]:
VALID_FRACTION = 0.10

session_meta = (
    events.groupby("session")
    .agg(end_ts=("ts", "max"), n_events=("aid", "size"))
    .reset_index()
    .sort_values("end_ts")
    .reset_index(drop=True)
)

split_idx = int(len(session_meta) * (1 - VALID_FRACTION))

train_sessions = set(session_meta.iloc[:split_idx]["session"])
valid_sessions = set(session_meta.iloc[split_idx:]["session"])

train_events = events[events["session"].isin(train_sessions)].copy()
valid_events = events[events["session"].isin(valid_sessions)].copy()

print(f"Train sessions: {len(train_sessions):,}")
print(f"Valid sessions: {len(valid_sessions):,}")
print("Train end:", train_events["datetime"].max())
print("Valid start:", valid_events["datetime"].min())


## 5. Validation task

For every validation session with at least two events, hide the final event.

`observed prefix -> held-out next item`


In [ ]:
validation_examples = []

for sid, group in valid_events.sort_values(["session", "ts", "position"]).groupby("session", sort=False):
    group = group.sort_values(["ts", "position"])
    if len(group) < 2:
        continue

    observed = group.iloc[:-1]
    target = group.iloc[-1]

    validation_examples.append({
        "session": sid,
        "observed_aids": observed["aid"].tolist(),
        "observed_types": observed["type"].tolist(),
        "target_aid": int(target["aid"]),
        "target_type": target["type"],
    })

valid_examples = pd.DataFrame(validation_examples)

print(f"Validation examples: {len(valid_examples):,}")
display(valid_examples["target_type"].value_counts())
valid_examples.head()


## 6. Metrics

In [ ]:
def hit_rate_at_k(recommendations, target, k=20):
    return float(target in recommendations[:k])

def reciprocal_rank_at_k(recommendations, target, k=20):
    top_k = recommendations[:k]
    try:
        return 1.0 / (top_k.index(target) + 1)
    except ValueError:
        return 0.0

def ndcg_at_k(recommendations, target, k=20):
    top_k = recommendations[:k]
    try:
        rank = top_k.index(target) + 1
        return 1.0 / math.log2(rank + 1)
    except ValueError:
        return 0.0

def evaluate_recommender(examples, recommend_fn, k=20):
    rows = []

    for row in examples.itertuples(index=False):
        recs = recommend_fn(row)

        rows.append({
            "target_type": row.target_type,
            "hit": hit_rate_at_k(recs, row.target_aid, k),
            "mrr": reciprocal_rank_at_k(recs, row.target_aid, k),
            "ndcg": ndcg_at_k(recs, row.target_aid, k),
        })

    result = pd.DataFrame(rows)

    overall = {
        "Recall@K": result["hit"].mean(),
        "HitRate@K": result["hit"].mean(),
        "MRR@K": result["mrr"].mean(),
        "NDCG@K": result["ndcg"].mean(),
    }

    by_type = (
        result.groupby("target_type")[["hit", "mrr", "ndcg"]]
        .mean()
        .rename(columns={"hit": "Recall@K", "mrr": "MRR@K", "ndcg": "NDCG@K"})
    )

    return overall, by_type


## 7. Baseline — Global popularity

In [ ]:
K = 20

global_popularity = train_events["aid"].value_counts().index.tolist()

def recommend_global(row, k=K):
    return global_popularity[:k]

global_overall, global_by_type = evaluate_recommender(valid_examples, recommend_global, K)
display(pd.Series(global_overall, name="Global Popularity"))
display(global_by_type)


## 8. Baseline — Event-weighted popularity

In [ ]:
EVENT_WEIGHTS = {
    "clicks": 1.0,
    "carts": 3.0,
    "orders": 6.0,
}

tmp = train_events.copy()
tmp["event_weight"] = tmp["type"].map(EVENT_WEIGHTS)

weighted_scores = tmp.groupby("aid")["event_weight"].sum().sort_values(ascending=False)
weighted_popularity = weighted_scores.index.tolist()

def recommend_weighted(row, k=K):
    return weighted_popularity[:k]

weighted_overall, weighted_by_type = evaluate_recommender(valid_examples, recommend_weighted, K)
display(pd.Series(weighted_overall, name="Event-Weighted Popularity"))
display(weighted_by_type)


## 9. Baseline — Recency-weighted popularity

In [ ]:
HALF_LIFE_DAYS = 3

tmp = train_events.copy()
max_ts = tmp["ts"].max()
half_life_ms = HALF_LIFE_DAYS * 24 * 60 * 60 * 1000

tmp["time_weight"] = np.exp(
    -np.log(2) * (max_ts - tmp["ts"]) / half_life_ms
)

tmp["score"] = tmp["type"].map(EVENT_WEIGHTS) * tmp["time_weight"]

recency_scores = tmp.groupby("aid")["score"].sum().sort_values(ascending=False)
recency_popularity = recency_scores.index.tolist()

def recommend_recency(row, k=K):
    return recency_popularity[:k]

recency_overall, recency_by_type = evaluate_recommender(valid_examples, recommend_recency, K)
display(pd.Series(recency_overall, name="Recency-Weighted Popularity"))
display(recency_by_type)


## 10. Baseline — Co-visitation

Build an item-to-item graph from nearby products inside the same session.


In [ ]:
PAIR_WINDOW = 20
MAX_NEIGHBORS = 100

pair_scores = defaultdict(Counter)

train_sorted = train_events.sort_values(["session", "ts", "position"])

for _, group in train_sorted.groupby("session", sort=False):
    aids = group["aid"].tolist()
    types = group["type"].tolist()

    for i, left_aid in enumerate(aids):
        upper = min(len(aids), i + PAIR_WINDOW + 1)

        for j in range(i + 1, upper):
            right_aid = aids[j]

            if left_aid == right_aid:
                continue

            distance_weight = 1.0 / (j - i)
            event_weight = EVENT_WEIGHTS.get(types[j], 1.0)
            score = distance_weight * event_weight

            pair_scores[left_aid][right_aid] += score
            pair_scores[right_aid][left_aid] += score

covisit_neighbors = {
    aid: [neighbor for neighbor, _ in scores.most_common(MAX_NEIGHBORS)]
    for aid, scores in pair_scores.items()
}

print(f"Items with neighbors: {len(covisit_neighbors):,}")


In [ ]:
def recommend_covisit(row, k=K):
    seen = set(row.observed_aids)
    candidate_scores = Counter()

    for recency_rank, aid in enumerate(reversed(row.observed_aids)):
        source_weight = 1.0 / (recency_rank + 1)

        for neighbor_rank, neighbor in enumerate(covisit_neighbors.get(aid, [])):
            if neighbor in seen:
                continue

            neighbor_weight = 1.0 / (neighbor_rank + 1)
            candidate_scores[neighbor] += source_weight * neighbor_weight

    recommendations = [aid for aid, _ in candidate_scores.most_common(k)]

    if len(recommendations) < k:
        for aid in global_popularity:
            if aid not in seen and aid not in recommendations:
                recommendations.append(aid)
                if len(recommendations) == k:
                    break

    return recommendations[:k]

covisit_overall, covisit_by_type = evaluate_recommender(valid_examples, recommend_covisit, K)
display(pd.Series(covisit_overall, name="Co-Visitation"))
display(covisit_by_type)


## 11. Baseline comparison

In [ ]:
comparison = pd.DataFrame([
    {"model": "global_popularity", **global_overall},
    {"model": "event_weighted_popularity", **weighted_overall},
    {"model": "recency_weighted_popularity", **recency_overall},
    {"model": "co_visitation", **covisit_overall},
]).set_index("model")

display(comparison.sort_values("Recall@K", ascending=False))

comparison[["Recall@K", "MRR@K", "NDCG@K"]].plot(kind="bar", figsize=(10, 5))
plt.title(f"OTTO Baselines @ {K}")
plt.ylabel("Score")
plt.xlabel("Model")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 12. Coverage

In [ ]:
def recommendation_coverage(examples, recommend_fn, k=20, max_examples=10_000):
    recommended = set()

    for row in examples.head(max_examples).itertuples(index=False):
        recommended.update(recommend_fn(row)[:k])

    catalog_size = train_events["aid"].nunique()

    return {
        "unique_recommended_items": len(recommended),
        "catalog_size": catalog_size,
        "catalog_coverage": len(recommended) / catalog_size,
    }

coverage = pd.DataFrame({
    "global_popularity": recommendation_coverage(valid_examples, recommend_global, K),
    "event_weighted_popularity": recommendation_coverage(valid_examples, recommend_weighted, K),
    "recency_weighted_popularity": recommendation_coverage(valid_examples, recommend_recency, K),
    "co_visitation": recommendation_coverage(valid_examples, recommend_covisit, K),
}).T

coverage


## 13. Inspect recommendations

In [ ]:
for row in valid_examples.head(10).itertuples(index=False):
    recs = recommend_covisit(row)

    print(f"Session:  {row.session}")
    print(f"Observed: {row.observed_aids[-10:]}")
    print(f"Target:   {row.target_aid} ({row.target_type})")
    print(f"Top-{K}:   {recs}")
    print(f"Hit:      {row.target_aid in recs}")
    print("-" * 100)


## 14. Save processed development data

In [ ]:
events_path = PROCESSED_DIR / f"otto_train_{N_SESSIONS // 1000}k_sessions.parquet"
valid_path = PROCESSED_DIR / f"otto_validation_{N_SESSIONS // 1000}k_sessions.parquet"

events.to_parquet(events_path, index=False)
valid_examples.to_parquet(valid_path, index=False)

print(events_path)
print(valid_path)


## Next

The next model notebook will move from heuristics to learned representations:

1. Item2Vec
2. BPR / implicit matrix factorization
3. Two-Tower retrieval
4. Candidate Recall@K comparison against co-visitation

After candidate generation is strong, we will add a separate ranking stage.
